# Tutorial 06b — Fused Attention **Backward**, Unpacked (a companion)

> A companion to **`tutorials_jupyter/06-fused-attention.ipynb`** (OpenAI's Triton
> **FlashAttention-2**), focused entirely on the **backward pass** — the four functions
> `_attn_bwd_preprocess`, `_attn_bwd_dkdv`, `_attn_bwd_dq`, `_attn_bwd`, plus the
> `torch.autograd` wiring. Notebook **06a** unpacked the forward and gave the backward a
> one-section summary; this one slows the backward all the way down: **one idea per section**,
> a figure, the exact lines from the tutorial, a small runnable NumPy twin, and an exercise.
>
> **Persona:** entry-to-intermediate practitioner (you know PyTorch autograd and the basic GPU
> memory hierarchy; you are learning Triton). Same voice as **06a** and course Chapters 16a–16d.

### Prerequisite

Read **06a** first (or at least its §2 online-softmax and §6 epilogue). This notebook assumes you
know that the forward pass saves two things for us: the output **`O`** and a per-row **logsumexp
`M`** — *not* the `(N, N)` score matrix. Everything here is built on those two tensors.

### How to use this notebook

The tutorial's real backward kernel targets **Hopper/Blackwell** GPUs. You almost certainly don't
have one, so the plan mirrors 06a:

- **Understand it on the CPU.** Every idea — the `D = rowsum(O ∘ dO)` collapse, recompute-from-`M`,
  and the two-axis `dK/dV` vs `dQ` split — is a tiny **NumPy simulation** checked against a
  finite-difference reference. These carry the exercises and run anywhere.
- **Read the kernels verbatim.** The four backward functions are quoted exactly from the tutorial
  and mapped line-by-line onto the NumPy twin.
- **Verify against autograd.** The last section checks the whole CPU backward against PyTorch's own
  gradients — the ultimate reference.

### Learning objectives

By the end you will be able to:

- Derive the five attention gradients (`dV, dP, dS, dQ, dK`) and see where the softmax Jacobian hides.
- Explain the **`D = rowsum(O ∘ dO)`** trick that collapses that Jacobian to one scalar per row
  (`_attn_bwd_preprocess`).
- Explain **recompute-not-store**: rebuilding `P` tile-by-tile from `M` alone, and why that drops
  backward memory from `O(N²)` to `O(N)`.
- Explain why FlashAttention-2 uses **two kernels on two axes** — `dK/dV` parallel over **key** blocks,
  `dQ` parallel over **query** blocks — and why that avoids atomics.
- Read `_attn_bwd_dkdv` and `_attn_bwd_dq` line by line, including the **transpose** and the folded
  `log2(e)` / `sm_scale` bookkeeping (`dq *= LN2`, `dk *= sm_scale`).

### Sources grounding this notebook

- The tutorial itself: `tutorials_jupyter/06-fused-attention.ipynb` (OpenAI Triton kernel team).
- Dao — [*FlashAttention-2*](https://arxiv.org/abs/2307.08691) (2023): the backward loop order and the
  `dK/dV`-over-keys / `dQ`-over-queries work partition.
- Dao, Fu, Ermon, Rudra, Ré — [*FlashAttention*](https://arxiv.org/abs/2205.14135) (2022): recomputation
  in the backward pass and the softmax-Jacobian identity.
- Companion: **06a** (this notebook's forward-pass sibling); course **Chapter 16b** writes the same
  algorithm in raw CUDA.


## 0. A map of the backward pass

The tutorial's monolith has **four** backward functions plus one method on the autograd wrapper.
Keep this table open as a legend — each row points to the section that opens it.

| In the tutorial cell | What it is | Covered in |
|---|---|---|
| `_attn_bwd_preprocess` | Compute `D = rowsum(O ∘ dO)` — one scalar per query row | §3 |
| `_attn_bwd_dkdv` | Inner loop for `dK`, `dV`; grid over **key** blocks, loops query rows | §5, §6 |
| `_attn_bwd_dq` | Inner loop for `dQ`; grid over **query** blocks, loops key columns | §5, §7 |
| `_attn_bwd` | Orchestrator: sets up pointers, calls `_attn_bwd_dkdv` then `_attn_bwd_dq`, handles causal | §5, §8 |
| `_attention.backward` | The `torch.autograd.Function` method that launches the three kernels | §10 |

**The shape convention** (same as 06a): tensors are `(Z, H, N_CTX, HEAD_DIM)` = (batch, heads,
sequence, head-dim). Below we work **one head at a time** as `Q, K, V` of shape `(N, d)`, exactly what
one `(z, h)` slice looks like. `dO` is the upstream gradient of the output `O`; our job is to turn it
into `dQ, dK, dV`.


## 1. What the forward pass left behind

The backward pass is a function of six tensors: the inputs `Q, K, V`, the output `O`, the upstream
gradient `dO`, and the saved logsumexp `M`. Crucially, **the `(N, N)` score/probability matrix is
gone** — the forward never stored it. Our first job is a plain forward that returns `O` *and* `M`,
so the rest of the notebook has something to differentiate and something to recompute from.

`M` is the **base-2 logsumexp** the kernel actually stores (06a §6): `M_i = log2(Σ_j 2^{S2_ij})`
where `S2 = S · log2(e)` are the base-2-scaled scores. Folding `log2(e)` in is what lets the kernel
use the fast `exp2` instruction; we carry the exact same convention so our `M` matches the tutorial's.


In [ ]:
import numpy as np

LOG2E = 1.4426950408889634    # = 1/ln(2) = log2(e); the tutorial writes 1.44269504

def attention_forward(Q, K, V, causal, sm_scale):
    """Plain attention forward. Returns (O, P, M):
      O : (N, d) output
      P : (N, N) softmax matrix — for teaching only; the real kernel never stores it
      M : (N,)   per-row base-2 logsumexp, the ONE thing (besides O) the forward saves
    """
    N, d = Q.shape
    S = (Q @ K.T) * sm_scale
    if causal:
        row, col = np.arange(N)[:, None], np.arange(N)[None, :]
        S = np.where(row >= col, S, -np.inf)          # keys after the query are invisible
    # standard softmax (ground-truth P and O)
    P = np.exp(S - S.max(axis=1, keepdims=True))
    P = P / P.sum(axis=1, keepdims=True)
    O = P @ V
    # base-2 logsumexp M — exactly what _attn_fwd's epilogue stores (m += log2(l))
    S2 = S * LOG2E
    m2 = np.max(np.where(np.isfinite(S2), S2, -np.inf), axis=1, keepdims=True)
    M = (m2 + np.log2(np.exp2(S2 - m2).sum(axis=1, keepdims=True))).ravel()
    return O, P, M

rng = np.random.default_rng(0)
N, d = 8, 4
Q = rng.standard_normal((N, d)); K = rng.standard_normal((N, d)); V = rng.standard_normal((N, d))
sm_scale = 1.0 / np.sqrt(d)
O, P, M = attention_forward(Q, K, V, causal=True, sm_scale=sm_scale)
print("O:", O.shape, " M:", M.shape, " (P is teaching-only, not saved by the kernel)")


## 2. The five gradients, on paper

Attention is three ops: `S = QKᵀ·scale`, then `P = softmax(S)` row-wise, then `O = P·V`. Differentiate
them back-to-front with the chain rule and you get exactly five gradients:

$$dV = P^\top dO, \qquad dP = dO\,V^\top, \qquad dS = P \circ \big(dP - \operatorname{rowsum}(P \circ dP)\big),$$
$$dQ = dS\,K \cdot \text{scale}, \qquad dK = dS^\top Q \cdot \text{scale}.$$

The only hard one is `dS`: softmax's Jacobian is dense, and the `rowsum(P ∘ dP)` term is where the
whole `(N, N)` matrix seems to be needed. §3 kills that term. `∘` is elementwise (Hadamard) product.

![Gradient chain](../course/figures/fig_t06b_grad_chain.svg)

Exercise A implements all five and checks them against **finite differences** — the honest test that
these formulas really are the gradient, not just internally consistent.


In [ ]:
# A finite-difference reference (numpy only) — the ground truth for the analytic gradients.
def _numeric_grad(f, X, eps=1e-5):
    """Central-difference gradient of scalar f() w.r.t. every entry of X (mutated in place)."""
    g = np.zeros_like(X); Xf = X.ravel(); gf = g.ravel()
    for k in range(Xf.size):
        old = Xf[k]
        Xf[k] = old + eps; fp = f()
        Xf[k] = old - eps; fm = f()
        Xf[k] = old
        gf[k] = (fp - fm) / (2 * eps)
    return g

def attention_backward_reference(Q, K, V, dO, causal, sm_scale):
    """The five analytic gradients, using D = rowsum(O ∘ dO) for the softmax-Jacobian term."""
    O, P, _ = attention_forward(Q, K, V, causal, sm_scale)
    dV = P.T @ dO
    dP = dO @ V.T
    D = (dO * O).sum(axis=1, keepdims=True)
    dS = P * (dP - D)
    dQ = (dS @ K) * sm_scale
    dK = (dS.T @ Q) * sm_scale
    return dQ, dK, dV

# self-test the reference against finite differences (loss = <dO, O>, so d loss/dX = the X-gradient)
_rng = np.random.default_rng(0); _N, _d = 6, 4
_Q = _rng.standard_normal((_N, _d)); _K = _rng.standard_normal((_N, _d))
_V = _rng.standard_normal((_N, _d)); _W = _rng.standard_normal((_N, _d))
_scale = 1.0 / np.sqrt(_d)
for _causal in (False, True):
    dQ, dK, dV = attention_backward_reference(_Q, _K, _V, _W, _causal, _scale)
    loss = lambda: float((attention_forward(_Q, _K, _V, _causal, _scale)[0] * _W).sum())
    assert np.allclose(dQ, _numeric_grad(loss, _Q), atol=1e-4)
    assert np.allclose(dK, _numeric_grad(loss, _K), atol=1e-4)
    assert np.allclose(dV, _numeric_grad(loss, _V), atol=1e-4)
    print(f"causal={_causal!s:5} | analytic dQ,dK,dV match finite differences")


In [ ]:
# EXERCISE A — implement the five attention gradients yourself.
# Return (dQ, dK, dV). You may call attention_forward to get O and P.
#   dV = Pᵀ·dO ,  dP = dO·Vᵀ ,  D = rowsum(O∘dO) ,  dS = P∘(dP − D) ,
#   dQ = dS·K·scale ,  dK = dSᵀ·Q·scale
def attention_backward(Q, K, V, dO, causal, sm_scale):
    raise NotImplementedError

# check against the finite-difference-validated reference above
rng = np.random.default_rng(1); N, d = 7, 5
Q = rng.standard_normal((N, d)); K = rng.standard_normal((N, d))
V = rng.standard_normal((N, d)); dO = rng.standard_normal((N, d))
sm_scale = 1.0 / np.sqrt(d)
for causal in (False, True):
    got = attention_backward(Q, K, V, dO, causal, sm_scale)
    ref = attention_backward_reference(Q, K, V, dO, causal, sm_scale)
    for a, b, name in zip(got, ref, ("dQ", "dK", "dV")):
        assert np.allclose(a, b, atol=1e-10), (causal, name, np.abs(a - b).max())
print("all five gradients correct for causal and non-causal")


<details>
<summary>▶ Show solution</summary>

```python
def attention_backward(Q, K, V, dO, causal, sm_scale):
    O, P, _ = attention_forward(Q, K, V, causal, sm_scale)
    dV = P.T @ dO
    dP = dO @ V.T
    D = (dO * O).sum(axis=1, keepdims=True)      # = rowsum(P ∘ dP), see §3
    dS = P * (dP - D)
    dQ = (dS @ K) * sm_scale
    dK = (dS.T @ Q) * sm_scale
    return dQ, dK, dV
```

</details>

## 3. `_attn_bwd_preprocess` — collapsing the softmax Jacobian

The awkward term in `dS` is `rowsum(P ∘ dP)`. Written out for row `i`:

$$\operatorname{rowsum}(P \circ dP)_i = \sum_j P_{ij}\,dP_{ij} = \sum_j P_{ij}\,(dO_i \cdot V_j)
= dO_i \cdot \Big(\sum_j P_{ij} V_j\Big) = dO_i \cdot O_i.$$

So the whole `(N, N)`-looking reduction is just `D_i = dO_i · O_i = rowsum(O ∘ dO)_i` — **one dot
product per row**, computable from tensors we already have (`O` and `dO`), with `P` nowhere in sight.
That is the entire job of `_attn_bwd_preprocess`:

```python
# _attn_bwd_preprocess — verbatim
o  = tl.load(O + ...)                     # this block of O
do = tl.load(DO + ...).to(tl.float32)     # this block of dO
delta = tl.sum(o * do, axis=1)            # D_i = rowsum(O ∘ dO)
tl.store(Delta + off_hz * N_CTX + off_m, delta)
```

It runs on its own tiny grid before the main backward kernel, writing one `Delta` vector. From then
on, `dS = P ∘ (dP − D)` — no rowsum over keys, no stored `P`.


In [ ]:
# EXERCISE B — implement _attn_bwd_preprocess and confirm D collapses the Jacobian.
def bwd_preprocess(O, dO):
    """Return D of shape (N,), where D_i = sum_j O_ij * dO_ij  (= rowsum(O ∘ dO))."""
    raise NotImplementedError

rng = np.random.default_rng(2); N, d = 6, 4
S = rng.standard_normal((N, N))
P = np.exp(S - S.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
V = rng.standard_normal((N, d)); dO = rng.standard_normal((N, d))
O = P @ V
dP = dO @ V.T

D = bwd_preprocess(O, dO)
assert D.shape == (N,)
# the identity: rowsum(O ∘ dO) equals the rowsum(P ∘ dP) term it replaces
assert np.allclose(D, (P * dP).sum(1)), "D must equal rowsum(P ∘ dP)"
# and it reproduces the full-Jacobian dS
dS_via_D = P * (dP - D[:, None])
dS_full  = P * (dP - (P * dP).sum(1, keepdims=True))
assert np.allclose(dS_via_D, dS_full)
print("one scalar per row replaces the (N x N) softmax Jacobian:  D =", np.round(D, 3))


<details>
<summary>▶ Show solution</summary>

```python
def bwd_preprocess(O, dO):
    return (O * dO).sum(axis=1)
# The identity rowsum(P∘dP) = rowsum(O∘dO) holds because dP = dO·Vᵀ and O = P·V:
#   Σ_j P_ij (dO_i · V_j) = dO_i · (Σ_j P_ij V_j) = dO_i · O_i.
```

</details>

## 4. Recompute, don't store: rebuilding `P` from `M`

`dS = P ∘ (dP − D)` still needs `P`. But the forward threw `P` away — it kept only `O` and the
per-row logsumexp `M`. FlashAttention's answer: **rebuild each `P` tile on the fly**, and `M` is
exactly the number that makes this a *single* `exp2`, with no second pass over the row.

Recall `M_i = log2(Σ_j 2^{S2_ij})` where `S2 = S · log2(e)`. Then

$$p_{ij} = 2^{\,S2_{ij} - M_i} = \frac{2^{S2_{ij}}}{\sum_{j'} 2^{S2_{ij'}}} = P_{ij}.$$

`M` folds the max-shift **and** the normalizer into one scalar per row, so recomputing `P` costs one
`qk` matmul and one `exp2` — no running max, no running sum. That is why the forward bothered to
store `M` (06a §6). Every backward kernel opens with this line: `p = tl.math.exp2(qk - m)`.

![Recompute not store](../course/figures/fig_t06b_recompute.svg)


In [ ]:
# EXERCISE C — rebuild P using ONLY the stored logsumexp M (no max, no sum).
def recompute_P(Q, K, M, causal, sm_scale):
    """Return P of shape (N, N), rebuilt as p = exp2(qk·scale·log2e − M), the way the
    backward kernels do. For causal, zero the (future) upper triangle after the exp2."""
    raise NotImplementedError

rng = np.random.default_rng(3); N, d = 8, 4
Q = rng.standard_normal((N, d)); K = rng.standard_normal((N, d)); V = rng.standard_normal((N, d))
sm_scale = 1.0 / np.sqrt(d)
for causal in (False, True):
    O, P, M = attention_forward(Q, K, V, causal, sm_scale)
    P_rebuilt = recompute_P(Q, K, M, causal, sm_scale)
    assert np.allclose(P_rebuilt, P, atol=1e-12), (causal, np.abs(P_rebuilt - P).max())
print("P rebuilt from M alone matches the forward's P — one exp2, no second pass over the row")


<details>
<summary>▶ Show solution</summary>

```python
def recompute_P(Q, K, M, causal, sm_scale):
    N = Q.shape[0]
    qk = (Q @ K.T) * sm_scale * LOG2E        # base-2-scaled scores S2 = S·log2(e)
    P = np.exp2(qk - M[:, None])             # M folds in both the max shift and the normalizer
    if causal:
        row, col = np.arange(N)[:, None], np.arange(N)[None, :]
        P = np.where(row >= col, P, 0.0)     # future keys contribute nothing
    return P
```

</details>

**See the trade, live.** The naive backward keeps the full `(N × N)` `P`; flash keeps only `O + M`
and rebuilds `P` tiles from `M`. Slide `N` and watch the memory gap open — this is why long-context
training is only feasible the flash way. *Renders when you run the cell in VS Code / Jupyter; on a
static viewer (GitHub) see the figure above.*

In [ ]:
# Interactive memory explorer for §4 — renders in Jupyter / VS Code (their output sandbox runs
# the embedded JavaScript). With no IPython frontend it degrades to a note; static viewers such
# as GitHub fall back to the figure above.
try:
    from IPython.display import HTML, display
    display(HTML(r"""<div id='t06b-mem'>
<div class='mm-title'>Interactive — the memory the backward pass refuses to spend</div>
<div class='mm-lede'>The naive backward stashes the full <code>(N x N)</code> probability matrix <b>P</b> for the gradient. FlashAttention stores only <b>O</b> <code>(N x d)</code> and the per-row logsumexp <b>M</b> <code>(N)</code>, then rebuilds each P tile on the fly. Slide N and watch the gap open (fp16, one head).</div>
<div class='mm-controls'>
<div class='mm-group'><span class='mm-lab'>N (sequence length)</span><div class='mm-step' id='mm-N'></div></div>
<div class='mm-group'><span class='mm-lab'>d (head dim)</span><div class='mm-step' id='mm-D'></div></div>
</div>
<div class='mm-bars' id='mm-bars'></div>
<div class='mm-note' id='mm-note'></div>
</div>
<style>
#t06b-mem{display:block;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;color:#1a2130;background:#ffffff;border:1px solid #d3dbe6;border-radius:14px;padding:18px 20px;max-width:820px;line-height:1.5;box-sizing:border-box}
#t06b-mem *{box-sizing:border-box}
#t06b-mem code{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:.9em;background:#eef2f7;padding:1px 4px;border-radius:4px;color:#1a2130}
#t06b-mem .mm-title{font-size:16px;font-weight:700;margin-bottom:4px}
#t06b-mem .mm-lede{font-size:13px;color:#5c6879;margin-bottom:14px}
#t06b-mem .mm-controls{display:flex;flex-wrap:wrap;gap:22px;margin-bottom:16px}
#t06b-mem .mm-lab{display:block;font-size:11px;letter-spacing:.04em;color:#5c6879;font-family:ui-monospace,Menlo,Consolas,monospace;margin-bottom:6px}
#t06b-mem .mm-step{display:inline-flex;align-items:center;border:1px solid #c3cad6;border-radius:8px;overflow:hidden}
#t06b-mem .mm-step button{border:0;background:#eef2f7;color:#1a2130;width:28px;height:30px;font-size:16px;cursor:pointer;font-family:inherit}
#t06b-mem .mm-step button:hover{background:#e0e6ef}
#t06b-mem .mm-step button:disabled{opacity:.35;cursor:not-allowed}
#t06b-mem .mm-step .mm-v{min-width:56px;text-align:center;font-variant-numeric:tabular-nums;font-weight:700;font-size:14px;font-family:ui-monospace,Menlo,Consolas,monospace}
#t06b-mem .mm-bars{display:flex;flex-direction:column;gap:12px;margin-bottom:6px}
#t06b-mem .mm-row{display:flex;align-items:center;gap:10px}
#t06b-mem .mm-rl{width:150px;font-size:12.5px;font-family:ui-monospace,Menlo,Consolas,monospace}
#t06b-mem .mm-track{flex:1;background:#f0f2f5;border-radius:7px;height:34px;position:relative;overflow:hidden;border:1px solid #e2e8f0}
#t06b-mem .mm-fill{height:100%;border-radius:6px 0 0 6px;display:flex;align-items:center;padding-left:10px;font-size:12px;font-weight:700;color:#fff;white-space:nowrap;min-width:2px}
#t06b-mem .mm-fill.naive{background:#c53030}
#t06b-mem .mm-fill.flash{background:#2f855a}
#t06b-mem .mm-note{font-size:12.5px;color:#5c6879;border-left:3px solid #2f855a;background:rgba(47,133,90,.08);padding:8px 12px;border-radius:0 6px 6px 0;margin-top:8px}
#t06b-mem .mm-note b{color:#1a2130}
</style>
<script>
(function(){
  var root=document.getElementById('t06b-mem');
  if(!root||root.dataset.init)return; root.dataset.init='1';
  var Ns=[512,1024,2048,4096,8192,16384], Ds=[32,64,128], BYTES=2;
  var st={ni:3,di:1};
  function stepper(host,arr,geti,seti,fmt){
    host.innerHTML='';
    var minus=document.createElement('button');minus.textContent='−';
    var val=document.createElement('span');val.className='mm-v';
    var plus=document.createElement('button');plus.textContent='+';
    function sync(){val.textContent=fmt(arr[geti()]);minus.disabled=geti()<=0;plus.disabled=geti()>=arr.length-1;}
    minus.onclick=function(){if(geti()>0){seti(geti()-1);render();}};
    plus.onclick=function(){if(geti()<arr.length-1){seti(geti()+1);render();}};
    host.appendChild(minus);host.appendChild(val);host.appendChild(plus);host._sync=sync;
  }
  function human(b){ if(b>=1073741824)return (b/1073741824).toFixed(2)+' GiB'; if(b>=1048576)return (b/1048576).toFixed(1)+' MiB'; if(b>=1024)return (b/1024).toFixed(1)+' KiB'; return b+' B'; }
  var sN=root.querySelector('#mm-N'), sD=root.querySelector('#mm-D'),
      bars=root.querySelector('#mm-bars'), note=root.querySelector('#mm-note');
  stepper(sN,Ns,function(){return st.ni;},function(v){st.ni=v;},function(x){return x.toLocaleString('en-US');});
  stepper(sD,Ds,function(){return st.di;},function(v){st.di=v;},function(x){return ''+x;});
  function render(){
    sN._sync(); sD._sync();
    var N=Ns[st.ni], d=Ds[st.di];
    var naive=N*N*BYTES, flash=N*(d+1)*BYTES, ratio=naive/flash;
    var wF=Math.max(0.6, 100*flash/naive);
    bars.innerHTML=
      '<div class="mm-row"><div class="mm-rl" style="color:#c53030">naive: store P<br><span style="color:#7c8aa0">N x N</span></div>'+
      '<div class="mm-track"><div class="mm-fill naive" style="width:100%">'+human(naive)+'</div></div></div>'+
      '<div class="mm-row"><div class="mm-rl" style="color:#2f855a">flash: store O+M<br><span style="color:#7c8aa0">N x d + N</span></div>'+
      '<div class="mm-track"><div class="mm-fill flash" style="width:'+wF+'%">'+human(flash)+'</div></div></div>';
    note.innerHTML='At <b>N='+N.toLocaleString('en-US')+'</b>, <b>d='+d+'</b>: flash keeps <b>'+ratio.toFixed(0)+'x less</b> for the backward pass ('+human(flash)+' vs '+human(naive)+', per head). '+
      'The missing P is never stored — every tile is rebuilt from M as <code>p = exp2(qk - M)</code>, one extra matmul. That is the <b>recompute-not-store</b> trade: a little compute to erase an O(N^2) memory bill.';
  }
  render();
})();
</script>"""))
except ImportError:
    print("Interactive widget needs a Jupyter/IPython frontend — see the static figure above.")

## 5. Two kernels, two axes — the FlashAttention-2 backward insight

Here is the idea the tutorial's structure is built around. `dK/dV` and `dQ` want **different**
parallelization axes:

- **`dK_j`, `dV_j`** gather contributions from *every query that attends to key block `j`*. Fix the
  **key** block, loop over query rows — one program owns the whole `dK_j`, `dV_j` sum. This is
  `_attn_bwd_dkdv`, gridded over **key** blocks.
- **`dQ_i`** gathers contributions from *every key that query block `i` attends to*. Fix the **query**
  block, loop over key columns — one program owns `dQ_i`. This is `_attn_bwd_dq`, gridded over
  **query** blocks.

Because each output slab is owned by exactly one program, there are **no atomics and no
cross-program accumulation** — the reason FA-2 is faster than FA-1's backward, which parallelized
only over query blocks and needed atomic adds into `dK/dV`.

![Two axes](../course/figures/fig_t06b_two_axes.svg)


### Why not one fused `_attn_bwd_dqdkdv`?

The obvious design is *one* kernel with *one* doubly-nested loop — which is exactly what the
FlashAttention-2 paper writes in **Algorithm 2**: an outer loop over key blocks `j`, an inner loop
over query blocks `i`, every gradient updated inside. So why does the real code split it in two?

Look at what each block sum reduces over:

| gradient | block sum | reduces along |
|---|---|---|
| `dV_j` | `Σᵢ Pᵢⱼᵀ · dOᵢ` | **query** blocks `i` |
| `dK_j` | `Σᵢ dSᵢⱼᵀ · Qᵢ` | **query** blocks `i` |
| `dQ_i` | `Σⱼ dSᵢⱼ · Kⱼ` | **key** blocks `j` |

`dK_j` and `dV_j` reduce along the **inner** loop. A program that owns key block `j` accumulates them
in registers across the whole sweep and stores once when the loop ends — Algorithm 2's line 18.

`dQ_i` reduces along the **outer** loop. Inside any single `j` iteration it is only *partially*
summed. That is why line 15 reads unlike its neighbours:

> **15:** *Load* `dQᵢ` **from HBM** to SRAM, then on chip, update `dQᵢ ← dQᵢ + dSᵢ⁽ʲ⁾ Kⱼ`, and
> **write back to HBM**.

Lines 12–16 all say "on chip". Line 15 is the one that round-trips HBM, on *every* inner iteration.
It is not an accumulator — it is a read-modify-write of memory shared with every other `j`.


In [ ]:
# Why one fused kernel fails: count the programs that WRITE each output slab.
from collections import Counter

N, BLOCK = 512, 128
n_blocks = N // BLOCK                         # Tr == Tc == 4 blocks along the sequence

def fused_writers(causal):
    """Algorithm 2's order: the grid IS the outer loop — one program per key block j."""
    dq_writes, dkv_writes = Counter(), Counter()
    for j in range(n_blocks):                 # grid: program j owns key block j
        for i in range(n_blocks):             # inner loop, on chip
            if causal and i < j:              # query block i cannot see key block j
                continue
            dq_writes[i] += 1                 # line 15: load dQ_i, add, store back -> HBM
        dkv_writes[j] += 1                    # line 18: dK_j/dV_j leave SRAM exactly once
    return dq_writes, dkv_writes

def split_writers(causal):
    """Triton's order: two passes, each output slab owned by exactly one program."""
    dq_writes, dkv_writes = Counter(), Counter()
    for j in range(n_blocks):                 # pass 1 = _attn_bwd_dkdv, grid over KEY blocks
        dkv_writes[j] += 1
    for i in range(n_blocks):                 # pass 2 = _attn_bwd_dq,   grid over QUERY blocks
        dq_writes[i] += 1
    return dq_writes, dkv_writes

for causal in (False, True):
    fq, fkv = fused_writers(causal)
    sq, skv = split_writers(causal)
    assert max(fkv.values()) == 1 and max(skv.values()) == 1   # dK/dV are safe under either order
    assert max(fq.values()) > 1                                # ...dQ is not: concurrent writers
    assert max(sq.values()) == 1                               # swapping the loops gives it one owner
    print(f"causal={causal!s:5} | fused: a dQ slab has up to {max(fq.values())} writers, "
          f"{sum(fq.values())} HBM round-trips -> needs atomicAdd")
    print(f"causal={causal!s:5} | split: a dQ slab has {max(sq.values())} writer,    "
          f"{sum(sq.values())} plain stores      -> lives in registers")
print("\ndK/dV reduce along the inner loop; dQ reduces along the loop we parallelize.")


Sequentially, line 15 is harmless. But the entire point of FA-2 is to **parallelize the outer
loop** — the grid is one program per key block. Now line 15 is a data race: all `Tc` programs
read-modify-write the same `dQᵢ` at once. Making that correct costs three things:

1. **`atomicAdd` into `dQ`** from every program, serializing the conflicting writes.
2. **A separate fp32 `dq_accum` buffer** in HBM — you cannot atomically accumulate `Tc` partial sums
   in fp16 without shedding precision — plus a cleanup pass to cast it back down.
3. **Nondeterminism.** Atomics land in scheduler order and float addition is not associative, so `dQ`
   differs bit-for-bit between runs. This is what `flash-attn`'s `deterministic=True` flag turns off.

Splitting the kernel declines all three. Give `dQ` its own pass with the loops **swapped** — the
program owns query block `i` and loops over key blocks `j` — and `dQᵢ` becomes an *inner*-loop
accumulator as well: `dq = tl.zeros([BLOCK_M2, HEAD_DIM], tl.float32)` in registers, one `tl.store`
at the end.

The bill arrives as **recompute**: both passes rebuild `qk = tl.dot(q, kT)` and `p` over the same
tiles, so `S` is formed twice. Cheap FLOPs traded for expensive HBM atomics — the same bargain §4
struck when it chose to recompute `P` from `M` rather than store it.


You cannot have it both ways. A single traversal of `S` can keep *either* the row reduction *or* the
column reduction register-resident, never both. **Two reduction axes, two passes** — that is the
whole reason `_attn_bwd_dqdkdv` does not exist.

One *launch*, though — not two. `_attn_bwd` runs on the grid `(N_CTX // BLOCK_N1, 1, BATCH · H)` and
reinterprets `pid` midway: the first phase reads it as a key block (`start_n = pid * BLOCK_N1`), the
second as a query block (`start_m = pid * BLOCK_M2`). That works only because the config picks
`BLOCK_N1 == BLOCK_M2 == 128`, so both phases want exactly the same number of programs. Keep that in
mind when you read `_attn_bwd` in §8 and find `pid` meaning two different things.


### First — what *is* this grid? `S = Q Kᵀ`, from two separate matrices

Before you drive the kernel widget: the grid it shows is **not one matrix**. It is the **score
matrix** `S = Q Kᵀ` — the product of two *separate* `(N × d)` arrays. `Q` supplies the **rows**,
`Kᵀ` (the transpose of `K`) supplies the **columns**. One **cell** `S[i, j]` is the dot product of
`Q` row `i` with `K` row `j`; one **tile** (query block × key block) is `Q_blk @ K_blkᵀ` — a single
`tl.dot` producing a `BLOCK_M × BLOCK_N` slab of scores.

Drive it below: click a cell of `S` (or a `Q` row / `Kᵀ` column), switch **element** vs **block**
view, resize `BLOCK`, and turn the causal mask on to watch the future cells drop out.


In [1]:
# Interactive Q / Kᵀ / S = Q Kᵀ decomposition explorer — renders in Jupyter / VS Code (their
# output sandbox runs the embedded JavaScript). With no IPython frontend it degrades to a note;
# static viewers such as GitHub fall back to the collapsed preview below.
try:
    from IPython.display import HTML, display
    display(HTML(r"""<div id='t06b-qk'>
<div class='qk-title'>Interactive — three separate matrices: Q, Kᵀ, and their product S = Q Kᵀ</div>
<div class='qk-lede'>The grid in the next widget is <b>not one matrix</b> — it is the <b>product</b> of two. Click any cell of <b>S</b> (or a row of <b>Q</b>, or a column of <b>Kᵀ</b>). In <b>element</b> view one cell of S is a dot product of a Q row and a K row; in <b>block</b> view one tile of S is <code>Q_blk @ K_blkᵀ</code> — exactly one <code>tl.dot</code>.</div>
<div class='qk-controls'>
<div class='qk-group'><span class='qk-lab'>view</span><div class='qk-btns'><button class='qk-b qk-mode qk-sel' data-m='block'>block tile — one tl.dot</button><button class='qk-b qk-mode' data-m='element'>element — one dot product</button></div></div>
<div class='qk-group'><span class='qk-lab'>BLOCK</span><div class='qk-step'><button id='qk-bm'>−</button><span class='qk-bv' id='qk-bv'></span><button id='qk-bp'>+</button></div></div>
<div class='qk-group'><span class='qk-lab'>causal mask</span><div class='qk-btns'><button class='qk-b qk-cau' data-c='1'>on</button><button class='qk-b qk-cau qk-sel' data-c='0'>off</button></div></div>
</div>
<div class='qk-body'>
<div class='qk-corner'>
  <div class='qk-cap qk-dim'>N = 8, d = 4</div>
  <div class='qk-cap qk-kcap'>Kᵀ &nbsp;(d × N)&nbsp; — keys →</div>
  <div class='qk-sp'></div>
  <div class='qk-kt' id='qk-kt'></div>
  <div class='qk-q' id='qk-q'></div>
  <div class='qk-s' id='qk-s'></div>
  <div class='qk-cap qk-qcap'>Q &nbsp;(N × d)&nbsp; — queries ↓</div>
  <div class='qk-cap qk-scap'>S = Q Kᵀ &nbsp;(N × N scores)</div>
</div>
<div class='qk-readout' id='qk-readout'></div>
</div>
</div>
<style>
#t06b-qk{display:block;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;color:#1a2130;background:#ffffff;border:1px solid #d3dbe6;border-radius:14px;padding:18px 20px;max-width:960px;line-height:1.5;box-sizing:border-box}
#t06b-qk *{box-sizing:border-box}
#t06b-qk code{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:.9em;background:#eef2f7;padding:1px 4px;border-radius:4px;color:#1a2130}
#t06b-qk .qk-title{font-size:16px;font-weight:700;margin-bottom:4px}
#t06b-qk .qk-lede{font-size:13px;color:#5c6879;margin-bottom:14px}
#t06b-qk .qk-controls{display:flex;flex-wrap:wrap;gap:18px;margin-bottom:16px}
#t06b-qk .qk-lab{display:block;font-size:11px;letter-spacing:.04em;color:#5c6879;font-family:ui-monospace,Menlo,Consolas,monospace;margin-bottom:6px}
#t06b-qk .qk-btns{display:flex;flex-wrap:wrap;gap:6px}
#t06b-qk .qk-b{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:12px;color:#1a2130;background:#eef2f7;border:1px solid #d3dbe6;border-radius:7px;padding:6px 10px;cursor:pointer}
#t06b-qk .qk-b:hover{border-color:#b3c0d1}
#t06b-qk .qk-b.qk-sel{background:#dd6b20;border-color:#dd6b20;color:#ffffff}
#t06b-qk .qk-step{display:inline-flex;align-items:center;border:1px solid #c3cad6;border-radius:8px;overflow:hidden}
#t06b-qk .qk-step button{border:0;background:#eef2f7;color:#1a2130;width:28px;height:30px;font-size:16px;cursor:pointer;font-family:inherit}
#t06b-qk .qk-step button:hover{background:#e0e6ef}
#t06b-qk .qk-step button:disabled{opacity:.35;cursor:not-allowed}
#t06b-qk .qk-bv{min-width:34px;text-align:center;font-weight:700;font-size:14px;font-family:ui-monospace,Menlo,Consolas,monospace}
#t06b-qk .qk-body{display:flex;flex-wrap:wrap;gap:26px;align-items:flex-start}
#t06b-qk .qk-corner{display:grid;grid-template-columns:auto auto;gap:6px 14px;align-items:start;justify-items:start}
#t06b-qk .qk-cap{font-size:11px;font-family:ui-monospace,Menlo,Consolas,monospace;color:#5c6879}
#t06b-qk .qk-kcap{color:#2f855a;font-weight:700}
#t06b-qk .qk-qcap{color:#2b6cb0;font-weight:700}
#t06b-qk .qk-scap{color:#dd6b20;font-weight:700}
#t06b-qk .qk-q{display:grid;grid-template-columns:22px repeat(4,26px);grid-auto-rows:26px;gap:2px}
#t06b-qk .qk-kt{display:grid;grid-template-columns:repeat(8,26px);grid-auto-rows:26px;gap:2px}
#t06b-qk .qk-s{display:grid;grid-template-columns:repeat(8,26px);grid-auto-rows:26px;gap:2px}
#t06b-qk .qk-idx{display:flex;align-items:center;justify-content:center;font-size:9.5px;color:#8b97a8;font-family:ui-monospace,Menlo,Consolas,monospace}
#t06b-qk .qk-c{width:26px;height:26px;border-radius:3px;border:1px solid transparent;cursor:pointer}
#t06b-qk .qk-qc{background:#eaf2fb}
#t06b-qk .qk-qc.alt{background:#dbe8f6}
#t06b-qk .qk-kc{background:#e7f6ec}
#t06b-qk .qk-kc.alt{background:#d6eede}
#t06b-qk .qk-sc{background:#f2f4f7}
#t06b-qk .qk-sc.alt{background:#e7ebf1}
#t06b-qk .qk-sc.fut{background:repeating-linear-gradient(45deg,rgba(124,138,160,.22) 0 4px,transparent 4px 8px)}
#t06b-qk .qk-qc.band{background:rgba(43,108,176,.24);border-color:#2b6cb0}
#t06b-qk .qk-kc.band{background:rgba(47,133,90,.24);border-color:#2f855a}
#t06b-qk .qk-sc.tile{background:rgba(221,107,32,.22);border-color:#dd6b20}
#t06b-qk .qk-sc.elmark{box-shadow:inset 0 0 0 2px #b45309}
#t06b-qk .qk-qc.sel{background:#2b6cb0;border-color:#2b6cb0}
#t06b-qk .qk-kc.sel{background:#2f855a;border-color:#2f855a}
#t06b-qk .qk-sc.el{background:#dd6b20;border-color:#b45309}
#t06b-qk .qk-readout{flex:1;min-width:310px}
#t06b-qk .qk-fact{font-size:13px;margin:0 0 10px}
#t06b-qk .qk-code{background:#f4f6fa;border:1px solid #d3dbe6;border-radius:9px;padding:12px;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:12px;line-height:1.8;white-space:pre;overflow-x:auto;color:#1a2130}
#t06b-qk .qk-qb{color:#2b6cb0;font-weight:700}
#t06b-qk .qk-kb{color:#2f855a;font-weight:700}
#t06b-qk .qk-sb{color:#dd6b20;font-weight:700}
#t06b-qk .qk-cmt{color:#7c8aa0}
#t06b-qk .qk-note{font-size:12px;color:#5c6879;border-left:3px solid #dd6b20;background:rgba(221,107,32,.08);padding:7px 10px;border-radius:0 6px 6px 0;margin-top:12px}
#t06b-qk .qk-note b{color:#1a2130}
</style>
<script>
(function(){
  var root=document.getElementById('t06b-qk');
  if(!root||root.dataset.init)return; root.dataset.init='1';
  var N=8, D=4, Bopts=[1,2,4];
  var st={mode:'block', B:2, i:3, j:2, causal:false};
  var elQ=root.querySelector('#qk-q'), elK=root.querySelector('#qk-kt'),
      elS=root.querySelector('#qk-s'), out=root.querySelector('#qk-readout');
  var bMinus=root.querySelector('#qk-bm'), bPlus=root.querySelector('#qk-bp'), bVal=root.querySelector('#qk-bv');

  root.querySelectorAll('.qk-mode').forEach(function(b){
    b.onclick=function(){st.mode=b.getAttribute('data-m');render();};});
  root.querySelectorAll('.qk-cau').forEach(function(b){
    b.onclick=function(){st.causal=b.getAttribute('data-c')==='1';render();};});
  bMinus.onclick=function(){var k=Bopts.indexOf(st.B); if(k>0){st.B=Bopts[k-1];render();}};
  bPlus.onclick=function(){var k=Bopts.indexOf(st.B); if(k<Bopts.length-1){st.B=Bopts[k+1];render();}};

  function mk(cls,txt,title,r,c){
    var d=document.createElement('div'); d.className=cls;
    if(txt!=null) d.textContent=txt;
    if(title) d.title=title;
    if(r!=null||c!=null) d.onclick=function(){ if(r!=null)st.i=r; if(c!=null)st.j=c; render(); };
    return d;
  }
  function sl(b,B){ return (b*B)+':'+((b+1)*B); }

  function render(){
    var B=st.B, br=Math.floor(st.i/B), bc=Math.floor(st.j/B), blk=(st.mode==='block');
    root.querySelectorAll('.qk-mode').forEach(function(x){x.classList.toggle('qk-sel',x.getAttribute('data-m')===st.mode);});
    root.querySelectorAll('.qk-cau').forEach(function(x){x.classList.toggle('qk-sel',(x.getAttribute('data-c')==='1')===st.causal);});
    bVal.textContent=B; bMinus.disabled=(Bopts.indexOf(B)<=0); bPlus.disabled=(Bopts.indexOf(B)>=Bopts.length-1);

    elQ.innerHTML='';
    for(var r=0;r<N;r++){
      elQ.appendChild(mk('qk-idx',r,'query row '+r,null,null));
      for(var c=0;c<D;c++){
        var cq='qk-c qk-qc';
        if(Math.floor(r/B)%2===1) cq+=' alt';
        if(blk && Math.floor(r/B)===br) cq+=' band';
        if(!blk && r===st.i) cq+=' sel';
        elQ.appendChild(mk(cq,null,'Q['+r+','+c+']',r,null));
      }
    }
    elK.innerHTML='';
    for(var c2=0;c2<N;c2++) elK.appendChild(mk('qk-idx',c2,'key '+c2,null,null));
    for(var k=0;k<D;k++){
      for(var c3=0;c3<N;c3++){
        var ck='qk-c qk-kc';
        if(Math.floor(c3/B)%2===1) ck+=' alt';
        if(blk && Math.floor(c3/B)===bc) ck+=' band';
        if(!blk && c3===st.j) ck+=' sel';
        elK.appendChild(mk(ck,null,'Kᵀ['+k+','+c3+'] = K['+c3+','+k+']',null,c3));
      }
    }
    elS.innerHTML='';
    for(var r2=0;r2<N;r2++){
      for(var c4=0;c4<N;c4++){
        var cs='qk-c qk-sc';
        if((Math.floor(r2/B)+Math.floor(c4/B))%2===1) cs+=' alt';
        if(st.causal && c4>r2) cs+=' fut';
        if(blk && Math.floor(r2/B)===br && Math.floor(c4/B)===bc) cs+=' tile';
        if(blk && r2===st.i && c4===st.j) cs+=' elmark';
        if(!blk && r2===st.i && c4===st.j) cs+=' el';
        elS.appendChild(mk(cs,null,'S['+r2+','+c4+']',r2,c4));
      }
    }
    out.innerHTML=readout(B,br,bc,blk);
  }

  function readout(B,br,bc,blk){
    var head='<p class="qk-fact">Two separate arrays live in HBM: <b class="qk-qb">Q ('+N+' × '+D+')</b> and <b class="qk-kb">K ('+N+' × '+D+')</b>. <b class="qk-sb">S</b> is their product — and the kernel <i>never stores it</i>.</p>';
    var cls, why;
    if(blk){
      if(bc>br){ cls='future — skipped'; why='Every (query, key) pair in this tile is in the future, so the causal kernel never loads it.'; }
      else if(bc===br){ if(B===1){ cls='diagonal — visible'; why='With BLOCK = 1 a tile is one (query, key) pair; the diagonal pair is visible.'; }
                        else { cls='diagonal — masked'; why='Some pairs inside this tile are future. It is the one tile per program that pays a mask.'; } }
      else { cls='off-band — mask-free'; why='Every pair in this tile is visible. No mask, full speed.'; }
      return head+
        '<div class="qk-code"><span class="qk-cmt"># block view — one tile of S = one tl.dot</span>\n'+
        '<span class="qk-sb">S['+sl(br,B)+', '+sl(bc,B)+']</span> = <span class="qk-qb">Q['+sl(br,B)+', :]</span> @ <span class="qk-kb">K['+sl(bc,B)+', :]</span>ᵀ\n'+
        '      ('+B+' × '+B+')   =    ('+B+' × '+D+')     @     ('+D+' × '+B+')</div>'+
        (st.causal
          ? '<p class="qk-note"><b>Causal:</b> tile (Q'+br+', K'+bc+') is <b>'+cls+'</b>. '+why+' This is exactly the green / amber / grey classification in the widget below.</p>'
          : '<p class="qk-note">This '+B+'×'+B+' slab is <b>one cell</b> of the block grid in the widget below. Raise <code>BLOCK</code> and the tiles grow; the two matrices Q and K never change.</p>');
    }
    if(st.j>st.i){ cls='future — masked to −inf'; why='Query '+st.i+' comes before key '+st.j+', so it cannot attend to it.'; }
    else if(st.j===st.i){ cls='diagonal — visible'; why='A query attending to itself.'; }
    else { cls='visible'; why='Query '+st.i+' may attend to key '+st.j+'.'; }
    return head+
      '<div class="qk-code"><span class="qk-cmt"># element view — one cell of S = one dot product</span>\n'+
      '<span class="qk-sb">S['+st.i+', '+st.j+']</span> = Σ_k  <span class="qk-qb">Q['+st.i+', k]</span> · <span class="qk-kb">K['+st.j+', k]</span>     <span class="qk-cmt">(k = 0 … '+(D-1)+')</span>\n'+
      '        = dot( <span class="qk-qb">row '+st.i+' of Q</span> , <span class="qk-kb">row '+st.j+' of K</span> )\n'+
      '        <span class="qk-cmt"># row j of K == column j of Kᵀ</span></div>'+
      (st.causal
        ? '<p class="qk-note"><b>Causal:</b> S['+st.i+','+st.j+'] is <b>'+cls+'</b>. '+why+'</p>'
        : '<p class="qk-note">A “score” is nothing more than this dot product. <b>Q and K stay separate</b> — S is formed one tile at a time in registers, used, and dropped.</p>');
  }
  render();
})();
</script>"""))
except ImportError:
    print("Interactive widget needs a Jupyter/IPython frontend — see the static preview below.")

<details>
<summary>▸ Static preview (for GitHub / nbviewer and other viewers that strip JavaScript)</summary>

![Q, Kᵀ, and their product S = Q Kᵀ](../course/figures/fig_t06b_qk_matmul.svg)

</details>

So the widget below is indexed `q\k` with **Q down the rows and K across the columns**: choosing a
*key* block (`_attn_bwd_dkdv`) picks a **column** of `S`; choosing a *query* block (`_attn_bwd_dq`)
picks a **row**. The backward gradients flow through this same product — `dQ = dS·K` and
`dK = dSᵀ·Q` — so both kernels re-touch the separate `Q` and `K`, never a stored `S`.


**Explore both kernels, live.** Pick a kernel, pick the block it owns, toggle causal, and read the
sweep it performs. Notice how causal turns each sweep into *exactly one* masked diagonal block plus a
run of mask-free blocks — the backward echo of 06a's STAGE trick. *Renders in VS Code / Jupyter; on a
static viewer use the figure above.*

In [2]:
# Interactive work-partition explorer for §5 — renders in Jupyter / VS Code (their output sandbox
# runs the embedded JavaScript). With no IPython frontend it degrades to a note; a static viewer
# such as GitHub shows only this code, so the figure and prose above stand on their own there.
try:
    from IPython.display import HTML, display
    display(HTML(r"""<div id='t06b-axes'>
<div class='ax-title'>Interactive — who computes which gradient, and over what?</div>
<div class='ax-lede'>FlashAttention-2 runs <b>two</b> backward kernels on <b>different</b> parallel axes. Pick a kernel, pick the block it owns, toggle causal. Green = <b>swept, mask-free</b>; amber = <b>diagonal, masked</b>; grey = <b>skipped (future)</b>. Each program writes one gradient slab alone — no atomics.</div>
<div class='ax-controls'>
<div class='ax-group'><span class='ax-lab'>kernel</span><div class='ax-btns'><button class='ax-b ax-mode ax-sel' data-mode='dkdv'>_attn_bwd_dkdv → dK, dV</button><button class='ax-b ax-mode' data-mode='dq'>_attn_bwd_dq → dQ</button></div></div>
<div class='ax-group'><span class='ax-lab' id='ax-sellab'>key block it owns</span><div class='ax-btns' id='ax-blocks'></div></div>
<div class='ax-group'><span class='ax-lab'>mode</span><div class='ax-btns'><button class='ax-b ax-caus ax-sel' data-c='1'>causal</button><button class='ax-b ax-caus' data-c='0'>non-causal</button></div></div>
</div>
<div class='ax-body'>
<div class='ax-gridwrap'><div class='ax-grid' id='ax-grid'></div>
<div class='ax-legend'><span class='ax-lg'><span class='ax-sw act'></span>swept · mask-free</span><span class='ax-lg'><span class='ax-sw mask'></span>diagonal · masked</span><span class='ax-lg'><span class='ax-sw skip'></span>future · skipped</span><span class='ax-lg'><span class='ax-sw own'></span>this program's column/row</span></div>
</div>
<div class='ax-readout' id='ax-readout'></div>
</div>
</div>
<style>
#t06b-axes{display:block;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;color:#1a2130;background:#ffffff;border:1px solid #d3dbe6;border-radius:14px;padding:18px 20px;max-width:940px;line-height:1.5;box-sizing:border-box}
#t06b-axes *{box-sizing:border-box}
#t06b-axes code{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:.9em;background:#eef2f7;padding:1px 4px;border-radius:4px;color:#1a2130}
#t06b-axes .ax-title{font-size:16px;font-weight:700;margin-bottom:4px}
#t06b-axes .ax-lede{font-size:13px;color:#5c6879;margin-bottom:14px}
#t06b-axes .ax-controls{display:flex;flex-wrap:wrap;gap:18px;margin-bottom:14px}
#t06b-axes .ax-lab{display:block;font-size:11px;letter-spacing:.04em;color:#5c6879;font-family:ui-monospace,Menlo,Consolas,monospace;margin-bottom:6px}
#t06b-axes .ax-btns{display:flex;flex-wrap:wrap;gap:6px}
#t06b-axes .ax-b{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:12px;color:#1a2130;background:#eef2f7;border:1px solid #d3dbe6;border-radius:7px;padding:6px 10px;cursor:pointer}
#t06b-axes .ax-b:hover{border-color:#b3c0d1}
#t06b-axes .ax-b.ax-sel{background:#2f855a;border-color:#2f855a;color:#ffffff}
#t06b-axes .ax-body{display:flex;flex-wrap:wrap;gap:22px;align-items:flex-start}
#t06b-axes .ax-grid{display:grid;grid-template-columns:64px repeat(4,58px);gap:4px}
#t06b-axes .ax-hd{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:10px;color:#7c8aa0;display:flex;align-items:center;justify-content:center;text-align:center}
#t06b-axes .ax-c{height:44px;border-radius:6px;border:1px solid transparent;display:flex;align-items:center;justify-content:center;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:11px;font-weight:600;background:#f0f2f5;color:#9aa6b6}
#t06b-axes .ax-c.bg{background:#f0f2f5;color:#c2ccd8}
#t06b-axes .ax-c.futbg{background:rgba(124,138,160,.09);color:#c2ccd8}
#t06b-axes .ax-c.act{background:rgba(47,133,90,.16);color:#2f855a;border-color:#2f855a}
#t06b-axes .ax-c.mask{background:repeating-linear-gradient(45deg,rgba(221,107,32,.18) 0 5px,transparent 5px 10px);color:#b45309;border:1px dashed #dd6b20}
#t06b-axes .ax-c.skip{background:rgba(124,138,160,.14);color:#9aa6b6}
#t06b-axes .ax-c.own{box-shadow:inset 0 0 0 2px #2b6cb0}
#t06b-axes .ax-legend{display:flex;flex-wrap:wrap;gap:6px 14px;margin-top:12px;max-width:320px}
#t06b-axes .ax-lg{display:flex;align-items:center;gap:6px;font-size:11.5px;color:#5c6879}
#t06b-axes .ax-sw{width:13px;height:13px;border-radius:3px;border:1px solid #d3dbe6}
#t06b-axes .ax-sw.act{background:rgba(47,133,90,.16);border-color:#2f855a}
#t06b-axes .ax-sw.mask{background:repeating-linear-gradient(45deg,rgba(221,107,32,.18) 0 3px,transparent 3px 6px);border:1px dashed #dd6b20}
#t06b-axes .ax-sw.skip{background:rgba(124,138,160,.14);border-color:#7c8aa0}
#t06b-axes .ax-sw.own{background:#fff;box-shadow:inset 0 0 0 2px #2b6cb0;border-color:#2b6cb0}
#t06b-axes .ax-readout{flex:1;min-width:300px}
#t06b-axes .ax-fact{font-size:13px;margin:0 0 10px}
#t06b-axes .ax-code{background:#f4f6fa;border:1px solid #d3dbe6;border-radius:9px;padding:12px;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:12px;line-height:1.7;white-space:pre;overflow-x:auto;color:#1a2130}
#t06b-axes .ax-g{color:#2f855a;font-weight:700}
#t06b-axes .ax-b2{color:#2b6cb0;font-weight:700}
#t06b-axes .ax-o{color:#b45309;font-weight:700}
#t06b-axes .ax-cmt{color:#7c8aa0}
#t06b-axes .ax-note{font-size:12px;color:#5c6879;border-left:3px solid #2b6cb0;background:rgba(43,108,176,.08);padding:6px 10px;border-radius:0 6px 6px 0;margin-top:12px}
</style>
<script>
(function(){
  var root=document.getElementById('t06b-axes');
  if(!root||root.dataset.init)return; root.dataset.init='1';
  var N=8,BLK=2,nB=4, state={mode:'dkdv',sel:1,causal:true};
  var grid=root.querySelector('#ax-grid'), blocks=root.querySelector('#ax-blocks'),
      readout=root.querySelector('#ax-readout'), sellab=root.querySelector('#ax-sellab');
  function mkbtns(){
    blocks.innerHTML='';
    for(var b=0;b<nB;b++){(function(b){
      var btn=document.createElement('button');
      btn.className='ax-b ax-blk'; btn.setAttribute('data-b',b);
      btn.textContent=(state.mode==='dkdv'?'K':'Q')+b+' ['+(b*BLK)+'-'+(b*BLK+BLK-1)+']';
      btn.onclick=function(){state.sel=b;render();}; blocks.appendChild(btn);})(b);}
  }
  root.querySelectorAll('.ax-mode').forEach(function(btn){
    btn.onclick=function(){state.mode=btn.getAttribute('data-mode');mkbtns();render();};});
  root.querySelectorAll('.ax-caus').forEach(function(btn){
    btn.onclick=function(){state.causal=btn.getAttribute('data-c')==='1';render();};});
  function mkhd(t){var d=document.createElement('div');d.className='ax-hd';d.textContent=t;return d;}
  function render(){
    root.querySelectorAll('.ax-mode').forEach(function(x){x.classList.toggle('ax-sel',x.getAttribute('data-mode')===state.mode);});
    root.querySelectorAll('.ax-caus').forEach(function(x){x.classList.toggle('ax-sel',(x.getAttribute('data-c')==='1')===state.causal);});
    root.querySelectorAll('.ax-blk').forEach(function(x){x.classList.toggle('ax-sel',(+x.getAttribute('data-b'))===state.sel);});
    sellab.textContent=state.mode==='dkdv'?'key block it owns':'query block it owns';
    var dk=state.mode==='dkdv', cz=state.causal, sel=state.sel;
    grid.innerHTML=''; grid.appendChild(mkhd('q\\k'));
    for(var c=0;c<nB;c++) grid.appendChild(mkhd('K'+c));
    for(var r=0;r<nB;r++){
      grid.appendChild(mkhd('Q'+r));
      for(var c2=0;c2<nB;c2++){
        var cell=document.createElement('div'); cell.className='ax-c';
        var owned = dk ? (c2===sel) : (r===sel);
        var cls;
        if(owned){ if(!cz){cls='act';} else if(r<c2){cls='skip';} else if(r===c2){cls='mask';} else {cls='act';} }
        else { cls=(cz&&r<c2)?'futbg':'bg'; }
        cell.classList.add(cls); if(owned) cell.classList.add('own');
        cell.textContent=(cls==='skip')?'':(cls==='mask'?'▨':(cls==='act'?'●':''));
        grid.appendChild(cell);
      }
    }
    var lo,hi;
    if(dk){
      lo = cz? sel : 0; hi = nB;
      var maskedNote = cz? '1 masked (diagonal Q'+sel+') + '+(hi-lo-1)+' mask-free' : (hi-lo)+' mask-free';
      readout.innerHTML=
        '<p class="ax-fact">One program owns <b class="ax-g">key block K'+sel+'</b> — key rows <b>['+(sel*BLK)+', '+((sel+1)*BLK)+')</b>. K'+sel+', V'+sel+' stay in SRAM; it sweeps <b>down the column</b> and writes <b class="ax-g">dK'+sel+', dV'+sel+'</b>.</p>'+
        '<div class="ax-code"><span class="ax-cmt"># _attn_bwd_dkdv : grid over KEY blocks (program_id -> start_n)</span>\n'+
        '<span class="ax-g">start_n</span> = '+(sel*BLK)+'      <span class="ax-cmt"># this key block</span>\n'+
        'query sweep = ['+(lo*BLK)+', '+N+')   <span class="ax-cmt"># queries that can see key '+(sel*BLK)+'</span>\n'+
        'num_steps  = '+(hi-lo)+'          <span class="ax-cmt"># '+maskedNote+'</span>\n\n'+
        '<span class="ax-cmt"># accumulate down the column, in registers:</span>\n'+
        'dv += pT @ do ;  dk += dsT @ qT</div>'+
        '<p class="ax-note"><b>Why key-parallel?</b> dK'+sel+' and dV'+sel+' each gather from <i>every</i> query that attends to this key. Fixing the key and looping queries lets <b>one</b> program own the whole sum — so it writes dK'+sel+'/dV'+sel+' once, no atomics.</p>';
    } else {
      lo = 0; hi = cz? (sel+1) : nB;
      var maskedNote2 = cz? '1 masked (diagonal K'+sel+') + '+(hi-lo-1)+' mask-free' : (hi-lo)+' mask-free';
      readout.innerHTML=
        '<p class="ax-fact">One program owns <b class="ax-b2">query block Q'+sel+'</b> — query rows <b>['+(sel*BLK)+', '+((sel+1)*BLK)+')</b>. Q'+sel+' stays in SRAM; it sweeps <b>across the row</b> and writes <b class="ax-b2">dQ'+sel+'</b>.</p>'+
        '<div class="ax-code"><span class="ax-cmt"># _attn_bwd_dq : grid over QUERY blocks (program_id -> start_m)</span>\n'+
        '<span class="ax-b2">start_m</span> = '+(sel*BLK)+'      <span class="ax-cmt"># this query block</span>\n'+
        'key sweep   = [0, '+(hi*BLK)+')   <span class="ax-cmt"># keys this query block can see</span>\n'+
        'num_steps  = '+(hi-lo)+'          <span class="ax-cmt"># '+maskedNote2+'</span>\n\n'+
        '<span class="ax-cmt"># accumulate across the row, in registers:</span>\n'+
        'dp = do @ vT ;  ds = p*(dp - D) ;  dq += ds @ kT</div>'+
        '<p class="ax-note"><b>Why query-parallel?</b> dQ'+sel+' gathers from every key this query attends to. Fixing the query and looping keys lets one program own dQ'+sel+' end to end — the mirror image of the dK/dV kernel, transposed.</p>';
    }
  }
  mkbtns(); render();
})();
</script>"""))
except ImportError:
    print("Interactive widget needs a Jupyter/IPython frontend — see the two-axes figure above.")

In [ ]:
# EXERCISE D — reproduce the sweep range each backward kernel performs, per block (causal & not).
# dkdv owns a KEY block and loops QUERY rows; dq owns a QUERY block and loops KEY columns.
def dkdv_query_range(start_n, block_n, n_ctx, causal):
    """(lo, hi) half-open range of query-row indices _attn_bwd_dkdv loops over for key block start_n.
       Causal: key n is visible only to queries m >= n, so the sweep starts at the block's own start."""
    raise NotImplementedError

def dq_key_range(start_m, block_m, n_ctx, causal):
    """(lo, hi) half-open range of key-column indices _attn_bwd_dq loops over for query block start_m.
       Causal: query m sees keys n <= m, so the sweep ends at the block's own end."""
    raise NotImplementedError

# key block at 256, query block at 256, block size 128, sequence length 1024
assert dkdv_query_range(256, 128, 1024, True)  == (256, 1024)   # queries at or after the key block
assert dkdv_query_range(256, 128, 1024, False) == (0, 1024)     # every query
assert dq_key_range(256, 128, 1024, True)      == (0, 384)      # keys up to the query block's end
assert dq_key_range(256, 128, 1024, False)     == (0, 1024)     # every key
# the diagonal block (256..384) is the single masked step in each causal sweep:
assert dkdv_query_range(256, 128, 1024, True)[0] == 256 == dq_key_range(256, 128, 1024, True)[1] - 128
print("backward sweep ranges OK — dK/dV walks queries >= its keys; dQ walks keys <= its queries")


<details>
<summary>▶ Show solution</summary>

```python
def dkdv_query_range(start_n, block_n, n_ctx, causal):
    return (start_n, n_ctx) if causal else (0, n_ctx)

def dq_key_range(start_m, block_m, n_ctx, causal):
    return (0, start_m + block_m) if causal else (0, n_ctx)
```

</details>

## 6. `_attn_bwd_dkdv` line by line — and the transpose

This is the inner loop that accumulates `dK_j`, `dV_j` for one key block, looping over query tiles.
Here is its body, verbatim:

```python
# _attn_bwd_dkdv — per query tile
qT = tl.load(qT_ptrs)                    # queries, loaded TRANSPOSED: (HEAD_DIM, BLOCK_M)
m = tl.load(M + offs_m)                  # the saved logsumexp for these queries
qkT = tl.dot(k, qT)                      # (BLOCK_N, BLOCK_M) — scores, transposed
pT = tl.math.exp2(qkT - m[None, :])      # recompute pᵀ from M (§4)
if MASK:                                 # only the diagonal block pays this
    mask = (offs_m[None, :] >= offs_n[:, None])
    pT = tl.where(mask, pT, 0.0)
do = tl.load(do_ptrs)
dv += tl.dot(pT.to(tl.float16), do)      # dV_j += pᵀ · dO
Di = tl.load(D + offs_m)
dpT = tl.dot(v, tl.trans(do)).to(tl.float32)   # dPᵀ = V · dOᵀ
dsT = pT * (dpT - Di[None, :])           # dSᵀ = pᵀ ∘ (dPᵀ − D)
dk += tl.dot(dsT.to(tl.float16), tl.trans(qT))  # dK_j += dSᵀ · Q
```

Line-for-line this is `dV = Pᵀ·dO`, `dP = dO·Vᵀ`, `dS = P∘(dP−D)`, `dK = dSᵀ·Q` from §2 — just
written key-major, with `pT[n, m] = P[query m, key n]`. Every `ᵀ` in that listing is **forced**, and
none of them costs a data movement. The rest of §6 says why.


### Why everything is transposed: the accumulator decides

This program owns one **key** block. It holds `dk` and `dv` — each `(BLOCK_N, HEAD_DIM)` — in
registers for the whole query loop, and writes them to HBM once, at the end. That single commitment
propagates backward through every line above.

Write `dV` out elementwise, and notice the contraction runs over `m`, the **queries**:

    dV[n, :] = Σ_m  P[m, n] · dO[m, :]

`tl.dot(A, B)` contracts the **inner** axis of `A` against the **outer** axis of `B`. The result has
to land as `(BLOCK_N, HEAD_DIM)` — that is the accumulator you already committed to. So `B` is `dO`
in its natural `(BLOCK_M, HEAD_DIM)` layout, and `A` must be `(BLOCK_N, BLOCK_M)`. That is `Pᵀ`. No
other assignment of shapes type-checks. `dK[n, :] = Σ_m dS[m, n] · Q[m, :]` pins `dsT` the same way.

And once `pT` and `dsT` are key-major, so is everything upstream of them — because the ops that
produce them are **elementwise**, and elementwise demands matching shapes:

    dsT = pT * (dpT − Di)    ⇒   dpT must be (BLOCK_N, BLOCK_M)
    pT  = exp2(qkT − m)      ⇒   qkT must be (BLOCK_N, BLOCK_M)

So the transpose is not a decision made at `qkT` and carried forward. It is a constraint pushed
**backward**, from the shape of the tile this program is responsible for.


### Why it is free: direct identities and transposed loads

None of this costs a data movement, because **the untransposed tile is never built and then
flipped.** Each transposed quantity is produced directly, by an identity:

| you want | identity | the line |
|----------|----------|----------|
| `Sᵀ`  | `(Q·Kᵀ)ᵀ = K·Qᵀ`   | `qkT = tl.dot(k, qT)` |
| `dPᵀ` | `(dO·Vᵀ)ᵀ = V·dOᵀ` | `dpT = tl.dot(v, tl.trans(do))` |

Same FLOPs, same single `tl.dot`. And `qT` arrives from `tl.load(qT_ptrs)` — a **transposed load**,
which is nothing but pointer arithmetic with the strides swapped. It reads the same bytes of `Q` out
of HBM that a normal load would; the re-layout happens on the shared-memory→register hop that feeds
the MMA, where Ampere and later have an `ldmatrix.trans` and get it for free.


### The transpose you must never write

Now imagine doing it the naive way: build `p` as `(BLOCK_M, BLOCK_N)`, then `tl.trans(p)` before the
`dv` dot. `p` has just come out of an MMA accumulator, and **an MMA accumulator layout is not
transposable in-register** — each thread is holding the wrong elements. The tile has to spill to
shared memory and be read back with a different pattern. You would pay that on the *largest* tile in
the loop (`BLOCK_M × BLOCK_N`, say 128×128, against 128×64 for the `HEAD_DIM` tiles), on *every*
iteration of the query loop.

Hence the rule this kernel actually follows:

> **Transpose the skinny `HEAD_DIM` operands at load time, where it is free.
> Never transpose the score-sized tile.**

Which is why `tl.trans` is not banned above — it appears twice, on `do` and on `qT`. Both are
`HEAD_DIM`-sized in one dimension, and both are freshly *loaded* tiles in a plain blocked layout,
not MMA outputs. Meanwhile `pT`, `dpT`, `dsT` — all `(BLOCK_N, BLOCK_M)` — are never transposed
anywhere. Exercise E does exactly one such tile.


In [ ]:
# EXERCISE E — one dK/dV tile, the transposed way (the body of _attn_bwd_dkdv).
# You are the program for a KEY block. For one query tile you are handed:
#   pT  : (BLOCK_N, BLOCK_M) recomputed probs, TRANSPOSED  (pT[n, m] = P[query m, key n])
#   doi : (BLOCK_M, d)  upstream grad dO for those queries
#   vj  : (BLOCK_N, d)  this key block's values
#   qi  : (BLOCK_M, d)  those queries
#   Di  : (BLOCK_M,)    the preprocess scalars D for those queries
# Return this tile's contribution (dv, dk), each (BLOCK_N, d). Mirror the kernel:
#   dv += pT @ do ;  dpT = v @ doᵀ ;  dsT = pT ∘ (dpT − D) ;  dk += dsT @ q
def dkdv_tile(pT, doi, vj, qi, Di):
    raise NotImplementedError

# one key block of 2 keys, all 4 queries in one tile, non-causal (no mask) -> one tile is the whole sum
rng = np.random.default_rng(4); N, d = 4, 3
Q = rng.standard_normal((N, d)); K = rng.standard_normal((N, d))
V = rng.standard_normal((N, d)); dO = rng.standard_normal((N, d))
sm_scale = 1.0 / np.sqrt(d)
O, P, M = attention_forward(Q, K, V, causal=False, sm_scale=sm_scale)
D = (dO * O).sum(1)
dQ_ref, dK_ref, dV_ref = attention_backward_reference(Q, K, V, dO, False, sm_scale)

keys = slice(0, 2)
dv, dk = dkdv_tile(P[:, keys].T, dO, V[keys], Q, D)
assert np.allclose(dv, dV_ref[keys]), np.abs(dv - dV_ref[keys]).max()
assert np.allclose(dk * sm_scale, dK_ref[keys]), np.abs(dk * sm_scale - dK_ref[keys]).max()
print("transposed dK/dV tile matches the reference (dk still needs its ·sm_scale epilogue, §9)")


<details>
<summary>▶ Show solution</summary>

```python
def dkdv_tile(pT, doi, vj, qi, Di):
    dv = pT @ doi                      # dV_j += pᵀ · dO
    dpT = vj @ doi.T                   # dPᵀ = V · dOᵀ
    dsT = pT * (dpT - Di[None, :])     # dSᵀ = pᵀ ∘ (dPᵀ − D)
    dk = dsT @ qi                      # dK_j += dSᵀ · Q  (scaled by sm_scale later, §9)
    return dv, dk
```

</details>

## 7. `_attn_bwd_dq` line by line — the mirror image

`dQ` is the transpose of the above: fix a **query** block, loop key columns, accumulate `dQ_i`. Now
nothing is transposed — the program is query-major:

```python
# _attn_bwd_dq — per key tile
kT = tl.load(kT_ptrs)                     # keys, transposed: (HEAD_DIM, BLOCK_N)
vT = tl.load(vT_ptrs)
qk = tl.dot(q, kT)                        # (BLOCK_M, BLOCK_N) scores
p  = tl.math.exp2(qk - m)                 # recompute p from M (m is (BLOCK_M, 1))
if MASK:                                  # diagonal block only
    mask = (offs_m[:, None] >= offs_n[None, :])
    p = tl.where(mask, p, 0.0)
dp = tl.dot(do, vT).to(tl.float32)        # dP = dO · Vᵀ
ds = p * (dp - Di[:, None])               # dS = p ∘ (dP − D)
dq += tl.dot(ds.to(tl.float16), tl.trans(kT))   # dQ_i += dS · K
```

Same five gradients, query-major. The `dq += tl.dot(ds, trans(kT))` gathers this query block's slice
of `dS·K` across all its key tiles. Two epilogue rescales finish the job — the subject of §9 — and
then `dQ_i` is written once. In the next section we assemble both loops into a full CPU backward and
check it end to end.


## 8. Putting it together — a full tiled backward on the CPU

Now both loops in one function, structurally the tutorial's `_attn_bwd`: parallel over key blocks for
`dK/dV`, parallel over query blocks for `dQ`, recomputing `p` from `M` on every tile, with the causal
sweep ranges from Exercise D. This is FlashAttention's backward with nothing hardware-specific — and
it matches the reference for both causal and non-causal attention, across block sizes.

![Causal backward](../course/figures/fig_t06b_causal_bwd.svg)


In [ ]:
def flash_backward_numpy(Q, K, V, dO, causal, sm_scale, BLOCK_M=16, BLOCK_N=16):
    """The two-kernel backward on the CPU — the algorithm _attn_bwd implements on the GPU."""
    N, d = Q.shape
    O, _, M = attention_forward(Q, K, V, causal, sm_scale)
    D = (dO * O).sum(axis=1)                                  # _attn_bwd_preprocess
    dQ = np.zeros((N, d)); dK = np.zeros((N, d)); dV = np.zeros((N, d))

    # --- _attn_bwd_dkdv: one program per KEY block, loop query rows (transposed) ---
    for j0 in range(0, N, BLOCK_N):
        kj, vj = K[j0:j0 + BLOCK_N], V[j0:j0 + BLOCK_N]
        bn = kj.shape[0]
        dk = np.zeros((bn, d)); dv = np.zeros((bn, d))
        i_lo = j0 if causal else 0                            # queries that can see this key block
        for i0 in range(i_lo, N, BLOCK_M):
            qi, doi = Q[i0:i0 + BLOCK_M], dO[i0:i0 + BLOCK_M]
            bm = qi.shape[0]
            Mi, Di = M[i0:i0 + BLOCK_M], D[i0:i0 + BLOCK_M]
            qkT = (kj @ qi.T) * sm_scale * LOG2E              # (bn, bm) transposed scores
            pT = np.exp2(qkT - Mi[None, :])                   # recompute pᵀ from M
            if causal:
                keys = j0 + np.arange(bn)[:, None]
                qrys = i0 + np.arange(bm)[None, :]
                pT = np.where(qrys >= keys, pT, 0.0)          # mask future (diagonal block)
            dv += pT @ doi
            dpT = vj @ doi.T
            dsT = pT * (dpT - Di[None, :])
            dk += dsT @ qi
        dK[j0:j0 + bn] = dk * sm_scale                        # ·sm_scale epilogue (§9)
        dV[j0:j0 + bn] = dv

    # --- _attn_bwd_dq: one program per QUERY block, loop key cols ---
    for i0 in range(0, N, BLOCK_M):
        qi, doi = Q[i0:i0 + BLOCK_M], dO[i0:i0 + BLOCK_M]
        bm = qi.shape[0]
        Mi, Di = M[i0:i0 + BLOCK_M], D[i0:i0 + BLOCK_M]
        dq = np.zeros((bm, d))
        j_hi = (i0 + bm) if causal else N                     # keys this query block can see
        for j0 in range(0, j_hi, BLOCK_N):
            kj, vj = K[j0:j0 + BLOCK_N], V[j0:j0 + BLOCK_N]
            bn = kj.shape[0]
            qk = (qi @ kj.T) * sm_scale * LOG2E
            p = np.exp2(qk - Mi[:, None])
            if causal:
                qrys = i0 + np.arange(bm)[:, None]
                keys = j0 + np.arange(bn)[None, :]
                p = np.where(qrys >= keys, p, 0.0)
            dp = doi @ vj.T
            ds = p * (dp - Di[:, None])
            dq += ds @ kj
        dQ[i0:i0 + bm] = dq * sm_scale
    return dQ, dK, dV

rng = np.random.default_rng(5); N, d = 32, 8
Q = rng.standard_normal((N, d)); K = rng.standard_normal((N, d))
V = rng.standard_normal((N, d)); dO = rng.standard_normal((N, d))
sm_scale = 1.0 / np.sqrt(d)
for causal in (False, True):
    for BM, BN in ((8, 8), (16, 8), (8, 16)):
        got = flash_backward_numpy(Q, K, V, dO, causal, sm_scale, BLOCK_M=BM, BLOCK_N=BN)
        ref = attention_backward_reference(Q, K, V, dO, causal, sm_scale)
        for a, b, name in zip(got, ref, ("dQ", "dK", "dV")):
            assert np.allclose(a, b, atol=1e-9), (causal, BM, BN, name, np.abs(a - b).max())
    print(f"causal={causal!s:5} | tiled two-kernel backward == reference (all block sizes)")


## 9. The scaling bookkeeping: `arg_k`, `dq *= LN2`, `dk *= sm_scale`

Two mysterious lines sit at the end of the tutorial's backward, and they are pure accounting for the
`exp2` trick. In the wrapper, before launching, the key is **pre-scaled**:

```python
RCP_LN2 = 1.4426950408889634   # = 1/ln(2) = log2(e)
arg_k = k * (sm_scale * RCP_LN2)   # fold BOTH sm_scale and log2(e) into K
```

so that `qk = q @ arg_kᵀ` already equals the base-2 scaled score `S2`, and `p = exp2(qk − M)` gives
the true probabilities directly. But folding those factors into `K` leaves the *output* gradients
scaled, so the kernel undoes it at the very end:

```python
dq *= LN2       # LN2 = ln(2);  cancels the log2(e) that arg_k baked into dQ
dk *= sm_scale  # dK was accumulated as dSᵀ·Q with no scale; put sm_scale back
```

`dV` needs no rescale — it never touched `K`. The exercise-free check below confirms the two rescales
are exactly the inverse factors, matching what our NumPy twin did explicitly (`dk * sm_scale`, and no
`log2(e)` fold because we scaled scores inline instead of pre-scaling `K`).


In [ ]:
# The tutorial's constants and why the two epilogue rescales are exact inverses of the fold.
LN2 = 0.6931471824645996        # ln(2)   — the kernel's `dq *= LN2`
RCP_LN2 = 1.4426950408889634    # 1/ln(2) — used to pre-scale arg_k (and == LOG2E)
assert np.isclose(LN2 * RCP_LN2, 1.0)                     # ln(2) · log2(e) = 1

# demonstrate the fold-and-undo end to end for dQ:
rng = np.random.default_rng(6); N, d = 8, 4
Q = rng.standard_normal((N, d)); K = rng.standard_normal((N, d))
V = rng.standard_normal((N, d)); dO = rng.standard_normal((N, d))
sm_scale = 1.0 / np.sqrt(d)
O, P, M = attention_forward(Q, K, V, False, sm_scale)
D = (dO * O).sum(1)

arg_k = K * (sm_scale * RCP_LN2)          # the wrapper's pre-scale
qk = Q @ arg_k.T                          # == S · log2(e)  (base-2 scaled)
p = np.exp2(qk - M[:, None])              # true probabilities, no separate scale needed
ds = p * ((dO @ V.T) - D[:, None])
dq_kernel = ds @ arg_k                    # accumulated against the PRE-SCALED K
dq_true = dq_kernel * LN2                 # <-- the `dq *= LN2` epilogue
dQ_ref, _, _ = attention_backward_reference(Q, K, V, dO, False, sm_scale)
assert np.allclose(dq_true, dQ_ref, atol=1e-12)
print("fold log2(e) into K, then `dq *= LN2` recovers the true dQ — the exp2 tax, paid back")


## 10. `_attention.backward` — wiring the three kernels into autograd

All of the above is launched from one `torch.autograd.Function.backward`. Trimmed to essentials:

```python
@staticmethod
def backward(ctx, do):
    q, k, v, o, M = ctx.saved_tensors            # <-- saved O and M, NOT the (N,N) scores
    dq = torch.empty_like(q); dk = torch.empty_like(k); dv = torch.empty_like(v)
    BATCH, N_HEAD, N_CTX = q.shape[:3]
    RCP_LN2 = 1.4426950408889634
    arg_k = k * (ctx.sm_scale * RCP_LN2)         # pre-scale K (§9)

    delta = torch.empty_like(M)
    _attn_bwd_preprocess[pre_grid](o, do, delta, ...)          # D = rowsum(O ∘ dO)  (§3)

    grid = (N_CTX // BLOCK_N1, 1, BATCH * N_HEAD)
    _attn_bwd[grid](q, arg_k, v, ctx.sm_scale, do, dq, dk, dv, # dK/dV then dQ  (§6–8)
                    M, delta, ..., CAUSAL=ctx.causal)
    return dq, dk, dv, None, None, None, None
```

The single line that captures the whole memory argument is **`ctx.saved_tensors → (q, k, v, o, M)`**.
A naive backward would stash the `(batch, heads, N, N)` attention matrix — gigabytes at long context.
FlashAttention saves only `O` and the vector `M`, and pays the §4 recompute to rebuild `p`. Backward
memory drops from `O(N²)` to `O(N)`; that trade — a little compute to avoid HBM traffic — is the
entire point of the algorithm, forward and backward alike.


## 11. Run it: check the CPU backward against PyTorch autograd

The NumPy twin is verified against finite differences and the analytic reference. The strongest
external check is PyTorch's own autograd: build a reference attention, let autograd produce
`dQ, dK, dV`, and confirm `flash_backward_numpy` reproduces them. This needs only `torch` (it runs on
CPU); without it the cell prints a notice and skips — the sections above already proved correctness.


In [ ]:
# Optional external check against PyTorch autograd. Guarded so it no-ops without torch.
try:
    import torch
    HAVE_TORCH = True
except Exception:                              # noqa: BLE001
    HAVE_TORCH = False
    print("PyTorch not available — skipping the autograd cross-check. "
          "The finite-difference and reference checks above already verified the backward.")

if HAVE_TORCH:
    torch.manual_seed(0)
    N, d = 64, 16
    sm_scale = 1.0 / (d ** 0.5)
    for causal in (False, True):
        q = torch.randn(N, d, dtype=torch.float64, requires_grad=True)
        k = torch.randn(N, d, dtype=torch.float64, requires_grad=True)
        v = torch.randn(N, d, dtype=torch.float64, requires_grad=True)
        s = (q @ k.T) * sm_scale
        if causal:
            s = s.masked_fill(~torch.tril(torch.ones(N, N, dtype=torch.bool)), float("-inf"))
        o = torch.softmax(s, dim=-1) @ v
        dO = torch.randn(N, d, dtype=torch.float64)
        o.backward(dO)

        dQ, dK, dV = flash_backward_numpy(q.detach().numpy(), k.detach().numpy(), v.detach().numpy(),
                                          dO.numpy(), causal, sm_scale, BLOCK_M=16, BLOCK_N=16)
        assert np.allclose(dQ, q.grad.numpy(), atol=1e-9)
        assert np.allclose(dK, k.grad.numpy(), atol=1e-9)
        assert np.allclose(dV, v.grad.numpy(), atol=1e-9)
        print(f"causal={causal!s:5} | flash_backward_numpy == torch.autograd gradients")


> **Going to production.** The real thing is three Triton kernels launched from
> `_attention.backward` (§10): `_attn_bwd_preprocess` on a `(N_CTX/128, BATCH·H)` grid, then
> `_attn_bwd` on a `(N_CTX/BLOCK_N1, 1, BATCH·H)` grid that runs `_attn_bwd_dkdv` then `_attn_bwd_dq`.
> On Hopper/Blackwell it adds the TMA / warp-specialization machinery from **06a §10**; on a consumer
> card it falls back to plain loads — the same algorithm you just ran on the CPU. To exercise it,
> run the tutorial's own `test_op` (which calls `torch.autograd` on the Triton `attention`) on a CUDA
> GPU with `pip install triton torch`.


## 12. Recap — the backward pass, now legible

You've opened every backward box in the 760-line cell:

| Concept | In the tutorial | You reproduced it in |
|---|---|---|
| Five attention gradients (`dV, dP, dS, dQ, dK`) | the algebra behind the kernels | Exercise A |
| `D = rowsum(O ∘ dO)` collapses the softmax Jacobian | `_attn_bwd_preprocess` | Exercise B |
| Recompute `P` from `M` (one `exp2`, no second pass) | `p = exp2(qk − m)` in every bwd kernel | Exercise C |
| `dK/dV` over **key** blocks, `dQ` over **query** blocks | `_attn_bwd`'s two loops | Exercise D + widget |
| Transposed `dK/dV` tile (`pᵀ, dpᵀ, dsᵀ`) | `_attn_bwd_dkdv` | Exercise E |
| Query-major `dQ` tile | `_attn_bwd_dq` | §7 + `flash_backward_numpy` |
| Full tiled backward, causal split | `_attn_bwd` | §8 `flash_backward_numpy` |
| Fold `log2(e)`/`sm_scale`, then `dq *= LN2`, `dk *= sm_scale` | wrapper + kernel epilogues | §9 |
| Save only `O`, `M`; recompute the rest → `O(N)` memory | `ctx.save_for_backward(q,k,v,o,M)` | §4, §10 |

**What you can now do:**

- Re-read the four backward functions in `tutorials_jupyter/06-fused-attention.ipynb` and know what
  every line is for, transpose and all.
- Explain why FA-2's backward is *exact* yet cheaper than storing the softmax: it recomputes `p` from
  `M` and partitions `dK/dV` vs `dQ` so no gradient needs atomics.
- Extend the §8 CPU twin — add dropout, an ALiBi/relative-position bias, or a different mask — using
  `attention_backward_reference` as your oracle.

**Where to go next:** course **Chapter 16b** writes this backward in raw CUDA; **06a** is the forward
sibling; **16c/16d** cover FlashAttention-3/4. You now have the whole fused-attention kernel open.
